In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv').drop(columns=['store room','floor_category','balcony'])

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type
0,flat,sector 36,0.82,3,2,New Property,850.0,0,0
1,flat,sector 89,0.95,2,2,New Property,1226.0,1,0
2,flat,sohna road,0.32,2,2,New Property,1000.0,0,0
3,flat,sector 92,1.60,3,4,Relatively New,1615.0,1,1
4,flat,sector 102,0.48,2,2,Relatively New,582.0,0,0


In [4]:
df['agePossession'] = df['agePossession'].replace(
    {
        'Relatively New':'new',
        'Moderately Old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
    }
)

In [5]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type
0,flat,sector 36,0.82,3,2,new,850.0,0,0
1,flat,sector 89,0.95,2,2,new,1226.0,1,0
2,flat,sohna road,0.32,2,2,new,1000.0,0,0
3,flat,sector 92,1.60,3,4,new,1615.0,1,1
4,flat,sector 102,0.48,2,2,new,582.0,0,0


In [6]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

C:\Users\farman ahmad\AppData\Local\Temp\ipykernel_28128\71934247.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'] = df['property_type'].replace({'flat':0,'house':1})


In [7]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type
0,0,sector 36,0.82,3,2,new,850.0,0,0
1,0,sector 89,0.95,2,2,new,1226.0,1,0
2,0,sohna road,0.32,2,2,new,1000.0,0,0
3,0,sector 92,1.60,3,4,new,1615.0,1,1
4,0,sector 102,0.48,2,2,new,582.0,0,0


In [9]:
new_df = pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True)

In [10]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [11]:
y_log = np.log1p(y)

In [12]:
y_log

0       0.598837
1       0.667829
2       0.277632
3       0.955511
4       0.392042
          ...   
3549    0.314811
3550    1.945910
3551    0.470004
3552    2.803360
3553    1.022451
Name: price, Length: 3554, dtype: float64

In [13]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [14]:
X_scaled = pd.DataFrame(X_scaled,columns=X.columns)

In [15]:
X_scaled

,property_type,bedRoom,bathroom,built_up_area,servant room,furnishing_type,sector_gwal pahari,sector_manesar,sector_sector 1,sector_sector 10,...,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sohna road,agePossession_old,agePossession_under construction
0,-0.517180,-0.074329,-0.874300,-0.831662,-0.747968,-0.668281,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
1,-0.517180,-0.877269,-0.874300,-0.522517,1.336956,-0.668281,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
2,-0.517180,-0.877269,-0.874300,-0.708333,-0.747968,-0.668281,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,4.561105,-0.602271,-0.290738
3,-0.517180,-0.074329,0.505173,-0.202684,1.336956,1.037949,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,5.877074,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
4,-0.517180,-0.877269,-0.874300,-1.052010,-0.747968,-0.668281,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3549,-0.517180,-0.877269,-0.874300,-1.093119,-0.747968,-0.668281,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
3550,1.933563,1.531549,1.194909,3.590095,1.336956,-0.668281,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
3551,-0.517180,-1.680208,-1.564036,-0.983768,-0.747968,1.037949,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,1.660383,-0.290738
3552,1.933563,1.531549,1.884645,2.983317,1.336956,-0.668281,-0.071348,-0.093805,-0.041123,-0.044424,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,1.660383,-0.290738


In [16]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [17]:
scores.mean(),scores.std()

(np.float64(0.8513047676938263), np.float64(0.016587714341053182))

In [18]:
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)

In [19]:
lr.fit(X_scaled,y_log)

LinearRegression()

In [20]:
ridge.fit(X_scaled,y_log)

Ridge(alpha=0.0001)

In [26]:
coef_df = pd.DataFrame(
    ridge.coef_.reshape(1, len(X.columns)),
    columns=X.columns
).stack().reset_index().drop(columns=['level_0']).rename(
    columns={'level_1': 'feature', 0: 'coef'}
)

In [27]:
coef_df

,feature,coef
0,property_type,0.119302
1,bedRoom,0.053706
2,bathroom,0.065660
3,built_up_area,0.210713
4,servant room,0.051890
...,...,...
106,sector_sector 95,-0.024845
107,sector_sector 99,-0.009905
108,sector_sohna road,-0.028473
109,agePossession_old,-0.008024


In [28]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(X_scaled)

# 3. Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.860
Method:                 Least Squares   F-statistic:                     198.3
Date:                Sat, 07 Nov 2026   Prob (F-statistic):               0.00
Time:                        22:06:26   Log-Likelihood:                 587.00
No. Observations:                3554   AIC:                            -950.0
Df Residuals:                    3442   BIC:                            -258.3
Df Model:                         111                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   

In [29]:
y_log.std()

0.5579613263072782

In [30]:
X_scaled['bedRoom'].std()

1.0001407162464122

In [31]:
0.21 * (0.557/1)

0.11697

In [32]:
np.expm1(0.030)

np.float64(0.030454533953516858)

In [33]:
2.4726962617564903e-05 * 100

0.0024726962617564905